# Beispiel zur hydraulischen Durchbildung einer Staustufe

Dieses Notebook fasst zwei zusammengehörige Aufgaben zusammen:

1. **Hydraulische Vorbemessung des Wehrs** (Poleni, (n‑1)‑Regel)
2. **Tosbeckenbemessung** (iterativer Workflow nach Bollrich / Peterka)

Lernziele:
- Rechengänge sehen;
- Kernannahmen verstehen;
- Größenordnungen prüfen können;
- Grenzen von Vereinfachungen erkennen.

In [ ]:
import math
from scipy.optimize import brentq
from pprint import pprint

---
# Teil 1 – Hydraulische Vorbemessung des Wehrs

## Inhalte

Es wird eine **überschlägige hydraulische Vorbemessung** eines vollregelnden Wehrs durchgeführt:

- Wahl eines plausiblen Bemessungsfalls
- Vorbemessung der erforderlichen lichten Wehrbreite
- Aufteilung in Wehrfelder
- Nachweis der **(n‑1)‑Regel** (DIN 19700)
- Grober Check von Freibord

## Gegebene Daten und Notation

**Poleni-Gleichung** für den freien Überfall:

$$
Q = c \cdot \frac{2}{3} \cdot \mu \cdot b_{\text{eff}} \cdot \sqrt{2g} \cdot h_{\text{ü}}^{3/2}
$$

**Effektive Überflussbreite** (Einschnürung durch Pfeiler):

$$
b_{\text{eff}} = n_{\text{aktiv}} \cdot \left(b_F - 2\,\xi_{\text{pf}}\,h_{\text{ü}}\right)
$$

*(Herleitung: $b_{\text{eff}} = b_{\text{ges}} - \sum b_{\text{pf}} - 2\,n_{\text{pf}}\,\xi_{\text{pf}}\,h_{\text{ü}}$  mit
$n_{\text{pf}} = (n_{\text{aktiv}}-1) + 2\cdot0{,}5 = n_{\text{aktiv}}$  Pfeileräquivalenten)*

| Symbol | Bedeutung |
|---|---|
| $Q$ | Durchfluss (m³/s) |
| $b_{\text{eff}}$ | wirksame Wehrbreite (m) |
| $h_{\text{ü}}$ | Überfallhöhe (m) |
| $c$ | Abminderungsbeiwert bei unvollkommenem Überfall (–) |
| $\mu$ | Abflussbeiwert (–) |
| $z_H$ | höchster zulässiger Aufstau des OW bei BHQ₁ (m) |
| $f$ | erforderlicher Freibord (m) |
| $n$ | Anzahl Wehrfelder |
| $b_F$ | Breite eines Wehrfelds (m) |
| $b_{\text{pf}}$ | Breite eines Pfeilers (m) |
| $\xi_{\text{pf}}$ | Einschnürungsbeiwert (–) |

## Eingangsgrößen

In [1]:
BHQ1 = 65.0    # m³/s  Bemessungshochwasser 1 (Regelfall)
BHQ2 = 85.0    # m³/s  Bemessungshochwasser 2 (Kontrollfall)

z_H  = 1.10    # m     höchster zulässiger OW-Aufstau bei BHQ1
f    = 0.50    # m     Freibord (DIN 19700: f ≥ 0,5 m bei BHQ1)

mu      = 0.64  # Abflussbeiwert (bewegliches Wehr, scharfkantig)
c       = 1.0   # Abminderungsbeiwert (vollkommener Überfall, Anfangswert)
xi_pf   = 0.10  # Einschnürungsbeiwert (abgerundete Pfeilerköpfe)

b_F  = 15.0              # m  gewählte Wehrfeldbreite (Anlehnung Hagneck)
b_pf = b_F * 0.225       # m  Pfeilerbreite (Drucksegmentverschluss)

g = 9.81  # m/s²

print(f"Pfeilerbreite b_pf = {b_pf:.2f} m")

Pfeilerbreite b_pf = 3.38 m


## Poleni-Berechnung – erforderliche Wehrbreite

**Entwurfsziel (DIN 19700, (n‑1)‑Regel):**  
Mit $(n-1)$ aktiven Wehrfeldern muss BHQ₁ abgeführt werden, ohne dass $h_{\text{ü}} > z_H$.

Auflösung der Poleni-Gleichung nach $b_{\text{eff}}$:

$$
b_{\text{eff,erf}} = \frac{Q_{\text{BHQ}_1}}{c \cdot \frac{2}{3} \cdot \mu \cdot \sqrt{2g} \cdot z_H^{3/2}}
$$

Auflösung nach $h_{\text{ü}}$ (für gegebene Breite):

$$
h_{\text{ü}} = \left(\frac{Q}{c \cdot \frac{2}{3} \cdot \mu \cdot \sqrt{2g} \cdot b_{\text{eff}}}\right)^{2/3}
$$

In [ ]:
## Definition von Hilfsfunktionen für Poleni-Solver

In [ ]:
C_pol = (2/3) * mu * math.sqrt(2 * g)  # kombinierter Poleni-Koeffizient ohne b und h_ue

def b_eff_von_Q(Q, h_ue, c_abd=1.0):
    """Erforderliche effektive Wehrbreite für gegebenen Durchfluss."""
    return Q / (c_abd * C_pol * h_ue**1.5)

def h_ue_von_Q(Q, b_eff, c_abd=1.0):
    """Überfallhöhe bei gegebenem Durchfluss und effektiver Breite."""
    return (Q / (c_abd * C_pol * b_eff))**(2/3)

def b_eff_n_felder(n_aktiv, h_ue):
    """Effektive Breite für n_aktiv nebeneinanderliegende Felder (Einschnürung berücksichtigt)."""
    return n_aktiv * (b_F - 2 * xi_pf * h_ue)


## Erforderliche Breite und Anzahl Wehrfelder

In [ ]:
# Gesamte b_eff für (n-1)-Fall muss b_eff_erf >= b_eff_von_Q(BHQ1, z_H) erfüllen
b_eff_erf = b_eff_von_Q(BHQ1, z_H, c)
b_F_eff   = b_F - 2 * xi_pf * z_H          # wirksame Feldbreite je Feld

n_minus1  = math.ceil(b_eff_erf / b_F_eff)  # benötigte Anzahl (n-1)-Felder
n         = n_minus1 + 1                     # Gesamtanzahl Wehrfelder

print(f"Poleni-Koeffizient C_pol = {C_pol:.4f}")
print(f"Erforderliche b_eff (n-1-Fall) = {b_eff_erf:.2f} m")
print(f"Wirksame Feldbreite b_F_eff    = {b_F_eff:.2f} m")
print(f"Benötigte (n-1)-Felder         = {n_minus1}")
print(f"Gewählte Wehrfeldanzahl n      = {n}")

## Wehrabmessungen und Breite

In [ ]:
n_pfeiler  = n - 1               # Zwischenpfeiler
B_licht    = n * b_F             # lichte Gesamtbreite (nur Felder)
B_pfeiler  = n_pfeiler * b_pf    # Summe Pfeilerbreiten
B_gesamt   = B_licht + B_pfeiler # Gesamtbreite inkl. Pfeiler

print(f"Anzahl Zwischenpfeiler : {n_pfeiler}")
print(f"Lichte Gesamtbreite    : {B_licht:.1f} m")
print(f"Summe Pfeilerbreiten   : {B_pfeiler:.2f} m")
print(f"Bauwerksbreite gesamt  : {B_gesamt:.2f} m")

## (n‑1)‑Regel -- Nachweis

**Bedingung:** Mit $(n-1)$ aktiven Feldern muss gelten: $h_{\text{ü}} \leq z_H$

Zusätzlich werden BHQ₁ und BHQ₂ mit allen $n$ Feldern geprüft.

In [ ]:
faelle = [
    ("BHQ1, alle n Felder",   BHQ1, n),
    ("BHQ1, (n-1) Felder",    BHQ1, n - 1),
    ("BHQ2, alle n Felder",   BHQ2, n),
]

print(f"{'Fall':<28} {'n_akt':>5} {'b_eff':>8} {'h_ü':>7} {'≤ z_H?':>8}")
print("-" * 60)
for label, Q, n_akt in faelle:
    # iterativ: b_eff hängt von h_ue ab, h_ue von b_eff → einmal fixpoint
    h_ue_iter = z_H
    for _ in range(20):
        beff = b_eff_n_felder(n_akt, h_ue_iter)
        h_ue_neu = h_ue_von_Q(Q, beff, c)
        if abs(h_ue_neu - h_ue_iter) < 1e-6:
            break
        h_ue_iter = h_ue_neu
    ok = "✓" if h_ue_iter <= z_H else "✗"
    print(f"{label:<28} {n_akt:>5} {beff:>8.2f} {h_ue_iter:>7.3f} {ok:>8}")

print(f"\n  z_H = {z_H:.2f} m  (max. zulässiger OW-Aufstau)")

## Freibord-Check

$$
\text{Kronenhöhe} = z_H + f \quad \text{mit} \quad f \geq 0{,}5\,\text{m (BHQ}_1\text{)}
$$

In [ ]:
# h_ue bei BHQ1, (n-1)-Felder (konservativster Fall)
h_ue_n1 = z_H
for _ in range(20):
    beff_n1   = b_eff_n_felder(n - 1, h_ue_n1)
    h_ue_n1   = h_ue_von_Q(BHQ1, beff_n1, c)

freibord_vorhanden = z_H - h_ue_n1
kronenhöhe         = z_H + f

print(f"h_ü bei BHQ1, (n-1)-Felder : {h_ue_n1:.3f} m")
print(f"Freibord über h_ü bis z_H  : {freibord_vorhanden:.3f} m")
print(f"Kronenhöhe (= z_H + f)     : {kronenhöhe:.2f} m über Wehrschwelle")
print(f"Freibordanforderung ≥ 0,5 m: {'✓' if f >= 0.5 else '✗'}")

---
# Teil 2: Tosbeckenbemessung

## Konzept

Ziel ist es, den **Wechselsprung** innerhalb des Tosbeckens zu erzwingen. Konstruktiver Freiheitsgrad: **Tosbeckeneintiefung** $e$ (in m).

**Einstaugrad** $\varepsilon$ muss im Zielbereich liegen:
$$
\varepsilon = \frac{h_u + e}{h_2} \in [1{,}05;\; 1{,}15]
$$

**Iterativer Algorithmus** (vgl. Vorlesungsfolie):

| Schritt | Inhalt |
|:---:|---|
| **A** | $e$ festlegen (Startwert) |
| **B** | $h_1$, $v_1$ aus Bernoulli-Gleichung (Energiehöhe $H_{\text{ges}}$) |
| **C** | $Fr_1 = v_1 / \sqrt{g\,h_1}$ |
| **D** | $4{,}5 \leq Fr_1 < 9{,}0$?  → Nein: $e$ anpassen |
| **E** | $h_u$ aus Manning–Strickler bzw. gemessen |
| **F** | $h_2 = \dfrac{h_1}{2}\left(\sqrt{1+8\,Fr_1^2}-1\right)$ (Bélanger) |
| **G** | $1{,}05 \leq \varepsilon \leq 1{,}15$?  → Nein: $e$ anpassen |
| **✓** | $l_T$ und $l_K$ berechnen |

## Zusätzliche Eingangsgrößen Tosbecken

Neben den Wehrparametern werden benötigt:

| Symbol | Bedeutung |
|---|---|
| $w$ | Wehrhöhe über UW-Sohle (m) |
| $k_{St}$ | Strickler-Beiwert der UW-Strecke (m¹/³/s) |
| $I_E$ | Sohlgefälle der UW-Strecke (–) |
| $B_u$ | Breite der UW-Strecke (m) |
| $H_{\text{ges}}$ | Gesamtenergielinie über Tosbeckensohle: $H_{\text{ges}} = z_H + w + e$ |

In [ ]:
w_wehr = 2.5   # m       Wehrhöhe über UW-Sohle
k_st   = 30.0  # m^1/3/s Strickler-Beiwert UW-Strecke (naturnahe Sohle)
I_E    = 0.001 # --      Sohlgefälle UW-Strecke
B_u    = 40.0  # m       Breite UW-Strecke (Rechteckprofil)

# Einheitsdurchfluss für Tosbeckenbemessung (alle n Felder aktiv)
h_ue_n = z_H
for _ in range(20):
    beff_n  = b_eff_n_felder(n, h_ue_n)
    h_ue_n  = h_ue_von_Q(BHQ1, beff_n, c)

q = BHQ1 / beff_n   # m²/s – Einheitsdurchfluss

print(f"BHQ1-Wehrbreite (n Felder): b_eff = {beff_n:.2f} m")
print(f"Einheitsdurchfluss q = BHQ1/b_eff  = {q:.3f} m²/s")

## Schritt E: Unterwassertiefe $h_u$ (Manning-Strickler)

Für ein **Rechteckprofil** gilt:
$$
Q = k_{St} \cdot A \cdot R^{2/3} \cdot I_E^{1/2}
\qquad \text{mit } A = B_u \cdot h_u,\quad R = \frac{B_u \cdot h_u}{B_u + 2\,h_u}
$$

Implizite Gleichung > numerische Lösung mittels folgender Funktion (und deren Aufrufen):

In [ ]:
def Q_manning(h, B, k, I):
    A = B * h
    R = (B * h) / (B + 2 * h)
    return k * A * R**(2/3) * math.sqrt(I)

# brentq sucht h in [0.01, 20] m mit Q_manning(h) = BHQ1
h_u = brentq(lambda h: Q_manning(h, B_u, k_st, I_E) - BHQ1, 0.01, 20.0)

print(f"Unterwassertiefe h_u = {h_u:.3f} m")
print(f"Probe: Q_Manning = {Q_manning(h_u, B_u, k_st, I_E):.2f} m³/s  (Soll: {BHQ1} m³/s)")

## Schritte A bis G: Iterative Tosbecken-Bemessung

### Schritt B Herleitung von $h_1$ und $v_1$

Energiehöhe über Tosbeckensohle (Bernoulli, vernachlässige $v_0$ da $v_0 < 1{,}0$ m/s):
$$
H_{\text{ges}} = z_H + w + e
$$

Am schießenden Querschnitt 1 (Tosbeckeneinlauf):
$$
H_{\text{ges}} = h_1 + \frac{v_1^2}{2g} = h_1 + \frac{q^2}{2g\,h_1^2}
$$

Dies ist eine kubische Gleichung in $h_1$ > numerische Lösung für die **kleine** (schießende) Wurzel.

### Kodieren des Arbeitsablaufs
Der Arbeitsablauf (Workflow) der Tosbeckenbemessung sowie die Ermittlung des k-Faktor in Abhängigkeit der Froude-Zahl wird in zwei Funktionen definiert, mit $e$ als Eingangsvariable:

In [ ]:
def tosbecken_check(e):
    """
    Führt Schritte A-G des Tosbecken-Workflows für gegebene Eintiefung e durch.
    Gibt (Fr1, epsilon, h1, h2, ok_Fr, ok_eps) zurück.
    """
    # Schritt B: h1 aus Bernoulli (kleine, schießende Wurzel)
    H_ges = z_H + w_wehr + e
    # Suche h1 < h_grenz (kritische Tiefe) = (q**2/g)^(1/3)
    h_krit = (q**2 / g)**(1/3)
    h1 = brentq(lambda h: h + q**2 / (2 * g * h**2) - H_ges, 1e-4, h_krit)
    v1 = q / h1

    # Schritt C: Froude-Zahl
    Fr1 = v1 / math.sqrt(g * h1)

    # Schritt D: Fr1-Check
    ok_Fr = 4.5 <= Fr1 < 9.0

    # Schritt F: konjugierte Tiefe h2 (Belanger)
    h2 = (h1 / 2) * (math.sqrt(1 + 8 * Fr1**2) - 1)

    # Schritt G: Einstaugrad
    eps = (h_u + e) / h2
    ok_eps = 1.05 <= eps <= 1.15

    return Fr1, eps, h1, h2, ok_Fr, ok_eps


# k-Faktor für Tosbeckenlänge (vgl. Peterka 1984 / USBR) 
def k_faktor(Fr1):
    if   Fr1 < 2.4:  return 4.8
    elif Fr1 < 4.0:  return 4.8 + (Fr1 - 2.4) / (4.0 - 2.4) * (5.8 - 4.8)
    elif Fr1 < 5.0:  return 5.8 + (Fr1 - 4.0) * (6.0 - 5.8)
    elif Fr1 < 6.0:  return 6.0
    elif Fr1 <= 11.0: return 6.13
    else:            return 6.0

### Iterieren über $e$
Die Eintiefung kann nun mittels der zuvor definierten Funktionen berechnet werden, basierend auf einem Startwert und einem Wertebereich:

In [ ]:
e_test   = 1.0   # m Startwert (Schritt A)
e_min    = 0.0   # m Suchbereich untere Grenze
e_max    = 3.0   # m Suchbereich obere Grenze
iterationen = []

print(f"{'Iter':>4}  {'e [m]':>7}  {'Fr1':>6}  {'Fr-OK':>6}  {'h1 [m]':>8}  {'h2 [m]':>8}  {'ε':>6}  {'ε-OK':>6}")
print("-" * 65)

for i in range(1, 15):
    Fr1, eps, h1, h2, ok_Fr, ok_eps = tosbecken_check(e_test)
    iterationen.append((e_test, Fr1, eps, h1, h2, ok_Fr, ok_eps))

    fr_sym  = "✓" if ok_Fr  else "✗"
    eps_sym = "✓" if ok_eps else "✗"
    print(f"{i:>4}  {e_test:>7.3f}  {Fr1:>6.2f}  {fr_sym:>6}  {h1:>8.4f}  {h2:>8.4f}  {eps:>6.3f}  {eps_sym:>6}")

    if ok_Fr and ok_eps:
        print("\n >>> Kriterien erfüllt: Iteration beendet.")
        break

    # Anpassungsstrategie (Bisektionslogik)
    if not ok_Fr:
        # Fr1 < 4.5 → e zu groß; Fr1 ≥ 9.0 → e zu klein
        if Fr1 < 4.5:
            e_max = e_test
        else:  # Fr1 >= 9.0
            e_min = e_test
    else:
        # ok_Fr = True, aber ok_eps = False
        if eps > 1.15:   # zu viel Einstau → e verringern
            e_max = e_test
        else:            # eps < 1.05 → zu wenig Einstau → e erhöhen
            e_min = e_test

    e_test = 0.5 * (e_min + e_max)  # Bisektion

# Endwerte speichern
e_opt  = e_test
Fr1_opt, eps_opt, h1_opt, h2_opt, _, _ = tosbecken_check(e_opt)

## Tosbeckenlänge $l_T$ und Kolkschutzlänge $l_K$

Nach Peterka (1958/1984):

$$
l_T = k \cdot h_2 \qquad l_K = 3{,}5 \cdot l_T
$$

| $Fr_1$ | 2,4 | 4 | 5 | 6–11 | 14 |
|---|---|---|---|---|---|
| $k$ | 4,8 | 5,8 | 6 | 6,13 | 6 |

Tosbecken- und Kolkschutzlänge können durch Aufrufen der oben definierten Funktion für die Abschätzung des k-Faktors berechnet werden:

In [ ]:
k   = k_faktor(Fr1_opt)
l_T = k * h2_opt
l_K = 3.5 * l_T

print(f"Optimierte Eintiefung   e   = {e_opt:.3f} m")
print(f"Schießende Tiefe        h1  = {h1_opt:.4f} m")
print(f"Froude-Zahl             Fr1 = {Fr1_opt:.2f}")
print(f"Konjugierte Tiefe       h2  = {h2_opt:.3f} m")
print(f"Unterwassertiefe        h_u = {h_u:.3f} m")
print(f"Einstaugrad             ε   = {eps_opt:.3f}  (Ziel: 1,05-1,15)")
print()
print(f"k-Faktor (Fr1={Fr1_opt:.1f})  k   = {k:.2f}")
print(f"Tosbeckenlänge          l_T = {l_T:.2f} m")
print(f"Kolkschutzlänge         l_K = {l_K:.2f} m")

## Zusammenfassung der hydraulischen Durchbildung inkl. Tosbeckenbemessung

In [ ]:

summary = {
    "Hydraulik_Wehr": {
        "BHQ1_m3s":              BHQ1,
        "BHQ2_m3s":              BHQ2,
        "z_H_m":                 z_H,
        "Freibord_m":            f,
        "n_Wehrfelder":          n,
        "b_F_m":                 b_F,
        "B_licht_m":             round(B_licht, 2),
        "B_gesamt_m":            round(B_gesamt, 2),
        "h_ue_BHQ1_nFelder_m":   round(h_ue_n, 3),
        "h_ue_BHQ1_n-1Felder_m": round(h_ue_n1, 3),
        "Kronenhöhe_m":          round(kronenhöhe, 2),
    },
    "Tosbecken": {
        "w_wehr_m":   w_wehr,
        "q_m2s":      round(q, 3),
        "h_u_m":      round(h_u, 3),
        "e_opt_m":    round(e_opt, 3),
        "h1_m":       round(h1_opt, 4),
        "Fr1":        round(Fr1_opt, 2),
        "h2_m":       round(h2_opt, 3),
        "epsilon":    round(eps_opt, 3),
        "k":          round(k, 2),
        "l_T_m":      round(l_T, 2),
        "l_K_m":      round(l_K, 2),
    },
}
pprint(summary)

---
## Grenzen der Vereinfachung

Dieses Notebook ersetzt **nicht**:

- standortbezogene Hydrologie (BHQ-Bestimmung)
- 1D/2D-Wasserstandsberechnungen für mehrere Abflüsse
- Nachweise für mehrere Last- und Ausfallfälle (a > 1, BHQ₂, ...)
- geotechnische Nachweise
- konstruktive Bemessung nach den jeweils einschlägigen Regeln
- Nachweise für Fischdurchgängigkeit, Sedimentmanagement und Betriebssicherheit
